# Mimicking Finance — single-panel replication

Runs the LSTM trade-direction replication on **one holdings parquet**. Toggle **per-fund** vs **global** LSTM.
Tuned for **CPU (~20 cores, limited RAM)**: per-fund training is parallelised across cores and fully mini-batched.
All logic lives in `pipeline.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import pipeline as P

## 1. Configure (CPU, memory-safe)
`parallel_backend='threading'` keeps ONE panel in shared memory (avoids the out-of-memory from process pools on a big panel). Lower `n_jobs`/`batch` if RAM is still tight. `step=1` = fully-overlapping windows (faithful, slow).

In [ ]:
cfg = P.Config(
    data_path='manager_holdings/master_batches_returnfiltered/panel_holdings_All_Funds_2002_master.parquet',
    model_mode='per_fund',        # 'per_fund' (parallel) or 'global'
    device='cpu',
    parallel_backend='threading', # SHARED memory -> safe for a big panel on a normal PC
    n_jobs=-1,                     # threads across cores (lower if you still see pressure)
    batch=4096,                    # train/val/predict mini-batch (lower e.g. 1024 for less RAM)
    downcast=True,                 # float32 + categoricals; also coerces pd.NA -> NaN
    keep_panel=False,              # free the panel after the run
    min_years=7, min_holdings=10, change_band=0.01,
    window_q=28, train_q=20, test_q=8, seq_len=8, step=8,
    max_epochs=25, hidden_cap=64, lr=3e-3,
    out_dir='outputs',
)
cfg

## 2. Run the full pipeline
(data prep → sequences → LSTM walk-forward → evaluation → figures)

In [ ]:
res = P.run(cfg)

## 3. Predictability (vs naive baseline)  — paper: LSTM 0.71, naive 0.52

In [ ]:
m = res['metrics']
pd.Series({k: round(m[k],4) for k in ['lstm_precision_pooled','naive_precision_pooled',
    'lstm_precision_fundavg','naive_precision_fundavg','n_funds','n_predictions']}).to_frame('value')

## 4. Table X — fund quintiles on predictability
Benchmark-adjusted cumulative abnormal return, 1–4 quarters. Q1=least predictable, Q5=most. Paper Q5−Q1 = −0.79% (t=−3.05).

In [ ]:
res['tables']['tableX'].round(4)

## 5. Table XI — correct vs incorrect positions  (paper diff −0.23%/qtr, t=−12.4)

In [ ]:
res['tables']['tableXI'].round(4)

## 6. Table XII — stock quintiles on cross-fund accuracy  (paper Q1−Q5 = +1.06%/qtr, t=5.74)

In [ ]:
res['tables']['tableXII'].round(4)

## 7. Figures

In [ ]:
res['figures']['precision_dist']

In [ ]:
res['figures']['stock_ls']

## 8. Compare per-fund vs global
Per-fund usually lifts LSTM precision **above** naive; global tends to **tie** it.

In [ ]:
cfg_g = P.Config(**{**{k:v for k,v in res['config'].items() if k!='col_map'},
                    'model_mode':'global','out_dir':'outputs_global'})
cfg_g.col_map = P.Config().col_map
res_g = P.run(cfg_g)
pd.DataFrame({
    'per_fund':[res['metrics']['lstm_precision_fundavg'], res['metrics']['naive_precision_fundavg'], res['metrics']['tableXII_Q1mQ5']],
    'global':  [res_g['metrics']['lstm_precision_fundavg'], res_g['metrics']['naive_precision_fundavg'], res_g['metrics']['tableXII_Q1mQ5']]},
    index=['LSTM prec (fund-avg)','Naive prec (fund-avg)','Table XII Q1-Q5']).round(4)